<a href="https://colab.research.google.com/github/rodrigoiyg27/TopicosEspeciales2026/blob/main/Clase7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# CSV oficial del paquete palmerpenguins (licencia CC0)
URL_PINGUINOS = (
    "https://raw.githubusercontent.com/allisonhorst/"
    "palmerpenguins/main/inst/extdata/penguins.csv"
)


pinguinos = pd.read_csv(URL_PINGUINOS)

# Verificación mínima: dimensiones esperadas (344 filas, 8 columnas)
print(pinguinos.shape)

(344, 8)


Frecuencias

In [ ]:
# Frecuencia absoluta: cuántos pingüinos hay de cada especie
frecuencia = pinguinos["species"].value_counts()

# Frecuencia relativa: qué porcentaje del total representa cada especie
porcentaje = (pinguinos["species"].value_counts(normalize=True) * 100).round(1)

tabla_frecuencias = pd.DataFrame({"frecuencia": frecuencia, "porcentaje": porcentaje})
print(tabla_frecuencias)

           frecuencia  porcentaje
species                          
Adelie            152        44.2
Gentoo            124        36.0
Chinstrap          68        19.8


Variables cuantitativas

In [ ]:
masa = pinguinos["body_mass_g"]

# Paso 1: el mínimo es 2700 g y el máximo 6300 g
print(masa.min(), masa.max())

# Paso 2: seis intervalos de 600 g cubren el rango completo
limites_de_clase = [2700, 3300, 3900, 4500, 5100, 5700, 6300]

# Paso 3: pd.cut asigna cada pingüino a su intervalo
# include_lowest=True hace que el primer intervalo incluya el mínimo
clases_de_masa = pd.cut(masa, bins=limites_de_clase, include_lowest=True)

# sort=False conserva el orden de los intervalos en vez de ordenar por frecuencia
frecuencia_por_clase = clases_de_masa.value_counts(sort=False)
print(frecuencia_por_clase)

2700.0 6300.0
body_mass_g
(2699.999, 3300.0]     40
(3300.0, 3900.0]      114
(3900.0, 4500.0]       73
(4500.0, 5100.0]       60
(5100.0, 5700.0]       43
(5700.0, 6300.0]       12
Name: count, dtype: int64


Medidas de posición

In [ ]:
# Cuartiles: tres cortes que dejan 25 % de los datos en cada tramo
print("Cuartiles\n", masa.quantile([0.25, 0.50, 0.75]))

# Deciles: nueve cortes, uno cada 10 %
print("Deciles\n",masa.quantile([i / 10 for i in range(1, 10)]))

# Un percentil concreto: el 90 % de los pingüinos pesa esto o menos
print("Percentil\n",masa.quantile(0.90))

Cuartiles
 0.25    3550.0
0.50    4050.0
0.75    4750.0
Name: body_mass_g, dtype: float64
Deciles
 0.1    3300.0
0.2    3475.0
0.3    3650.0
0.4    3800.0
0.5    4050.0
0.6    4300.0
0.7    4650.0
0.8    4950.0
0.9    5400.0
Name: body_mass_g, dtype: float64
Percentil
 5400.0


Tendencia Central

In [ ]:
# Largo de aleta de los siete primeros pingüinos con medida registrada
muestra_aletas = pinguinos["flipper_length_mm"].dropna().head(7)

print("Media: ", round(muestra_aletas.mean(), 2))
print("Mediana: ",muestra_aletas.median())
print("Moda: ", muestra_aletas.mode().tolist())

Media:  188.71
Mediana:  190.0
Moda:  [181.0, 195.0]


Ejemplo: Analizar la masa de los pingüinos

In [ ]:
from scipy import stats

# Serie con la masa corporal en gramos
masa = pinguinos["body_mass_g"]

# Pandas ignora los nulos (NaN) por defecto en estas tres operaciones
media = masa.mean()
mediana = masa.median()
moda = masa.mode()            # devuelve una Serie: puede haber más de una moda

# trim_mean NO ignora nulos: hay que quitarlos antes con dropna()
# proportiontocut=0.10 descarta el 10 % más bajo y el 10 % más alto
media_recortada = stats.trim_mean(masa.dropna(), proportiontocut=0.10)

print(f"Media:           {media:,.2f} g")
print(f"Mediana:         {mediana:,.2f} g")
print(f"Moda:            {moda.iloc[0]:,.2f} g")
print(f"Media recortada: {media_recortada:,.2f} g")

Media:           4,201.75 g
Mediana:         4,050.00 g
Moda:            3,800.00 g
Media recortada: 4,154.01 g


Tendencia central por grupo

In [ ]:
resumen_por_especie = (
    pinguinos
    .groupby("species")["body_mass_g"]               # una Serie de masa por especie
    .agg(n="count", media="mean", mediana="median")  # nombre_columna="función"
    .round(1)
)
print(resumen_por_especie)

# La moda es la única de estas medidas válida para variables categóricas
especie_mas_frecuente = pinguinos["species"].mode().iloc[0]   # "Adelie"

             n   media  mediana
species                        
Adelie     151  3700.7   3700.0
Chinstrap   68  3733.1   3700.0
Gentoo     123  5076.0   5000.0


Dispersión

In [ ]:
varianza = masa.var()               # ddof=1 por defecto: varianza muestral
desviacion = masa.std()             # raíz de la varianza, en gramos
rango = masa.max() - masa.min()     # depende solo de los dos extremos

# Cuartiles: Q1 deja el 25 % de los datos por debajo; Q3, el 75 %
q1 = masa.quantile(0.25)
q3 = masa.quantile(0.75)
iqr = q3 - q1                       # ancho del 50 % central de los datos

# MAD: mediana de las distancias absolutas a la mediana (medida robusta)
mad = (masa - masa.median()).abs().median()

# Coeficiente de variación: dispersión relativa, sin unidades
coef_variacion = desviacion / masa.mean() * 100

print(f"Varianza: {varianza:,.2f} g²")
print(f"Desviación estándar: {desviacion:,.2f} g")
print(f"Rango: {rango:,.0f} g")
print(f"IQR: {iqr:,.0f} g  (Q1={q1:,.0f}, Q3={q3:,.0f})")
print(f"MAD: {mad:,.0f} g")
print(f"CV: {coef_variacion:.2f} %")

Varianza: 643,131.08 g²
Desviación estándar: 801.95 g
Rango: 3,600 g
IQR: 1,200 g  (Q1=3,550, Q3=4,750)
MAD: 600 g
CV: 19.09 %


Resumen con describe

In [ ]:
columnas_numericas = [
    "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"
]
print(pinguinos[columnas_numericas].describe().round(2))

       bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g
count          342.00         342.00             342.00       342.00
mean            43.92          17.15             200.92      4201.75
std              5.46           1.97              14.06       801.95
min             32.10          13.10             172.00      2700.00
25%             39.22          15.60             190.00      3550.00
50%             44.45          17.30             197.00      4050.00
75%             48.50          18.70             213.00      4750.00
max             59.60          21.50             231.00      6300.00


Coeficiente de variación

In [ ]:
# El CV permite comparar variables medidas en unidades distintas (mm y g)
cv_por_variable = (
    pinguinos[columnas_numericas].std()
    / pinguinos[columnas_numericas].mean() * 100
).round(2)
print(cv_por_variable.sort_values(ascending=False))

body_mass_g          19.09
bill_length_mm       12.43
bill_depth_mm        11.51
flipper_length_mm     7.00
dtype: float64


Regla de Tukey

In [ ]:
def contar_atipicos(serie: pd.Series) -> int:
    """Cuenta los valores fuera de las cercas de Tukey (1.5 × IQR)."""
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    # Máscara booleana: True donde el valor queda fuera de los límites
    fuera_de_limites = (serie < limite_inferior) | (serie > limite_superior)
    return int(fuera_de_limites.sum())

# Sobre todo el dataset: ninguna variable tiene atípicos
print(pinguinos[columnas_numericas].agg(contar_atipicos))

# Dentro de cada especie: aparecen 6 valores inusuales
print(pinguinos.groupby("species")[columnas_numericas].agg(contar_atipicos))

bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
dtype: int64
           bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g
species                                                                 
Adelie                  0              1                  2            0
Chinstrap               0              0                  0            2
Gentoo                  1              0                  0            0


Matriz de correlación

In [ ]:
# Matriz de correlación entre todas las columnas numéricas
# corr() descarta los nulos par por par, sin borrar filas completas
matriz_pearson = pinguinos[columnas_numericas].corr()            # method="pearson"
matriz_spearman = pinguinos[columnas_numericas].corr(method="spearman")
matriz_kendall = pinguinos[columnas_numericas].corr(method="kendall")

# Las tres matrices, redondeadas a dos decimales para leerlas
print(matriz_pearson.round(2))
print(matriz_spearman.round(2))
print(matriz_kendall.round(2))

# El mismo par, aleta y masa, leído en cada una de las tres matrices
matrices = {"Pearson": matriz_pearson, "Spearman": matriz_spearman, "Kendall": matriz_kendall}
for nombre, matriz in matrices.items():
    # .loc[fila, columna] lee una celda de la matriz por sus nombres
    r_del_par = matriz.loc["flipper_length_mm", "body_mass_g"]
    print(f"{nombre}: {r_del_par:.3f}")

# Correlación entre dos columnas concretas
r_aleta_masa = pinguinos["flipper_length_mm"].corr(pinguinos["body_mass_g"])
print(f"r = {r_aleta_masa:.3f}   r² = {r_aleta_masa**2:.3f}")

                   bill_length_mm  bill_depth_mm  flipper_length_mm  \
bill_length_mm               1.00          -0.24               0.66   
bill_depth_mm               -0.24           1.00              -0.58   
flipper_length_mm            0.66          -0.58               1.00   
body_mass_g                  0.60          -0.47               0.87   

                   body_mass_g  
bill_length_mm            0.60  
bill_depth_mm            -0.47  
flipper_length_mm         0.87  
body_mass_g               1.00  
                   bill_length_mm  bill_depth_mm  flipper_length_mm  \
bill_length_mm               1.00          -0.22               0.67   
bill_depth_mm               -0.22           1.00              -0.52   
flipper_length_mm            0.67          -0.52               1.00   
body_mass_g                  0.58          -0.43               0.84   

                   body_mass_g  
bill_length_mm            0.58  
bill_depth_mm            -0.43  
flipper_length_mm       

Ordenar pares

In [ ]:
import numpy as np

# Máscara triangular superior: evita la diagonal y los pares repetidos
triangulo_superior = np.triu(np.ones(matriz_pearson.shape, dtype=bool), k=1)

pares_ordenados = (
    matriz_pearson
    .where(triangulo_superior)                # deja NaN fuera del triángulo
    .stack()                                  # matriz -> lista (var1, var2): r
    .dropna()                                 # quita los pares enmascarados
    .sort_values(key=abs, ascending=False)    # ordena por fuerza, sin importar el signo
)
print(pares_ordenados.round(3))

flipper_length_mm  body_mass_g          0.871
bill_length_mm     flipper_length_mm    0.656
                   body_mass_g          0.595
bill_depth_mm      flipper_length_mm   -0.584
                   body_mass_g         -0.472
bill_length_mm     bill_depth_mm       -0.235
dtype: float64


Reserva de datos

In [ ]:
# Mitad para explorar, elegida al azar; random_state fija la semilla
datos_exploracion = pinguinos.sample(frac=0.5, random_state=42)

# La otra mitad queda reservada: no se mira hasta tener la hipótesis escrita
datos_confirmacion = pinguinos.drop(datos_exploracion.index)


def mediana_por_especie_y_sexo(datos: pd.DataFrame) -> pd.DataFrame:
    """Tabla de masa mediana: una fila por especie, una columna por sexo."""
    return datos.pivot_table(
        index="species", columns="sex", values="body_mass_g", aggfunc="median"
    )


# 1) Explorar: aquí nace la hipótesis "los machos pesan más en cada especie"
print(mediana_por_especie_y_sexo(datos_exploracion))

# 2) Confirmar: la misma tabla sobre datos que no se habían mirado
tabla_confirmacion = mediana_por_especie_y_sexo(datos_confirmacion)
tabla_confirmacion["diferencia_g"] = (
    tabla_confirmacion["male"] - tabla_confirmacion["female"]
)
print(tabla_confirmacion)

sex        female    male
species                  
Adelie     3425.0  4025.0
Chinstrap  3500.0  3900.0
Gentoo     4712.5  5550.0
sex        female    male  diferencia_g
species                                
Adelie     3325.0  3950.0         625.0
Chinstrap  3600.0  3950.0         350.0
Gentoo     4675.0  5400.0         725.0
